In [1]:
import numpy as np
from sklearn.externals.array_api_extra.testing import override
from model_wrapper import *
import cuml.accel
from math import floor
cuml.accel.install()

# of Training Instances: 47
# of Testing Instances: 11
Current RAM usage: 298.73 MB


In [2]:
from sklearn.linear_model import SGDClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.decomposition import IncrementalPCA
from sklearn.kernel_approximation import RBFSampler

class SGDModel(Model):
    def __init__(self, anatomical_plane, fluid_sensitive=None, fat_suppression=None, pca_n_comp=20, rbf_n_comp=5, rbf_training_size=20):
        self.pca_n_comp = pca_n_comp
        self.rbf_n_comp = rbf_n_comp
        self.rbf_training_size = rbf_training_size
        self.inc_pca = None
        self.rbf_sampler = None
        self.model = MultiOutputClassifier(SGDClassifier(loss="log_loss", random_state=42))
        super().__init__(anatomical_plane, fluid_sensitive, fat_suppression, full_train=False)

    @override
    def batch_fit(self, training_folders: pd.Series, y: np.ndarray):
        if len(training_folders) <= self.pca_n_comp:
            self.inc_pca = IncrementalPCA(n_components=len(training_folders))
            n_batches = 1
        else:
            self.inc_pca = IncrementalPCA(n_components=self.pca_n_comp)
            n_batches = floor(len(training_folders) / self.pca_n_comp)

        self.rbf_sampler = RBFSampler(gamma=0.1, n_components=min(len(training_folders), self.rbf_n_comp), random_state=42)
        img_count = training_folders.apply(get_img_count)
        training_folders = training_folders[img_count >= MIN_IMG_COUNT]

        for folders_batch in np.array_split(training_folders, n_batches):
            X_batch = np.array([get_training_instance(f) for f in folders_batch])
            X_batch = np.reshape(X_batch, shape=(X_batch.shape[0], -1))
            self.inc_pca.partial_fit(X_batch)

        self.rbf_n_comp = min(self.rbf_n_comp, len(training_folders))
        self.rbf_training_size = min(self.rbf_training_size, len(training_folders))
        rbf_training_batch = np.array([get_training_instance(training_folders.iloc[i]) for i in range(self.rbf_training_size)])
        rbf_training_batch = np.reshape(
            rbf_training_batch,
            shape=(rbf_training_batch.shape[0], -1)
        )
        rbf_training_batch = self.inc_pca.transform(rbf_training_batch)
        self.rbf_sampler.fit(rbf_training_batch)

        all_classes = [np.array([0, 1]) for _ in range(y.shape[1])]
        batch_size = self.pca_n_comp
        for start_idx in range(0, len(training_folders), batch_size):
            end_idx = start_idx + batch_size
            folders_batch = training_folders.iloc[start_idx:end_idx]
            y_batch = y[start_idx:end_idx]

            X_batch = np.array([get_training_instance(f) for f in folders_batch])
            X_batch = np.reshape(X_batch, (X_batch.shape[0], -1))
            X_batch = self.inc_pca.transform(X_batch)
            X_batch = np.hstack((X_batch, self.rbf_sampler.transform(X_batch)))
            self.model.partial_fit(X_batch, y_batch, classes=all_classes)

    @override
    def predict_batch(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(x.shape[0], -1))
        x_reduced = self.inc_pca.transform(x)
        x_rbf = np.hstack((x_reduced, self.rbf_sampler.transform(x_reduced)))
        return self.model.predict(x_rbf)

    @override
    def predict_instance(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(1, -1))
        x_reduced = self.inc_pca.transform(x)
        x_rbf = np.hstack((x_reduced, self.rbf_sampler.transform(x_reduced)))
        pred_ = self.model.predict(x_rbf)
        return np.reshape(pred_, shape=(pred_.shape[1]))

In [3]:
ensemble = []

for p in planes:
    for i in range(2):
        ensemble.append(SGDModel(p, i, i))

Training Model: 
	Plane: Sagittal
	Fluid Sensitive: 0
	Fat Suppression: 0
	Training Size: (39,)
	Validation Size: (18,)
	AUC Score: 0.480982559107559
		ACL: 0.5416666666666666
		MCL: 0.30000000000000004
		Medial Meniscus: 0.375
		Lateral Meniscus: 0.2875
		Medial OA: 0.48461538461538456
		Lateral OA: 0.625
		PF OA: 0.6666666666666666
		Effusion: 0.5833333333333334
		Synovitis: 0.25
		Baker's: 0.48214285714285715
		Contusion: 0.46753246753246747
		Fracture: 0.7083333333333333

Training Model: 
	Plane: Sagittal
	Fluid Sensitive: 1
	Fat Suppression: 1
	Training Size: (33,)
	Validation Size: (17,)
	AUC Score: 0.4681449337699337
		ACL: 0.3472222222222222
		MCL: 0.8214285714285714
		Medial Meniscus: 0.5069444444444444
		Lateral Meniscus: 0.5642857142857143
		Medial OA: 0.25
		Lateral OA: 0.2857142857142857
		PF OA: 0.3560606060606061
		Effusion: 0.7803030303030303
		Synovitis: 0.5138888888888888
		Baker's: 0.3076923076923077
		Contusion: 0.5357142857142857
		Fracture: 0.3484848484848485

Tra

In [4]:
scores = Model.get_ensemble_auc_score(ensemble, 1)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

Model Score: 0.5873346560846561
	ACL: 0.5166666666666667
	MCL: 0.33333333333333337
	Medial Meniscus: 0.4285714285714286
	Lateral Meniscus: 0.6666666666666666
	Medial OA: 0.75
	Lateral OA: 0.6666666666666667
	PF OA: 0.5
	Effusion: 0.6607142857142857
	Synovitis: 0.4666666666666667
	Baker's: 0.7777777777777779
	Contusion: 0.5666666666666667
	Fracture: 0.7142857142857143


In [5]:
scores = Model.get_ensemble_auc_score(ensemble, 2)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

Model Score: 0.5753637566137567
	ACL: 0.33333333333333337
	MCL: 0.2777777777777778
	Medial Meniscus: 0.3928571428571429
	Lateral Meniscus: 0.7333333333333334
	Medial OA: 0.875
	Lateral OA: 0.7777777777777778
	PF OA: 0.3571428571428572
	Effusion: 0.7142857142857143
	Synovitis: 0.39999999999999997
	Baker's: 0.8333333333333334
	Contusion: 0.5666666666666667
	Fracture: 0.6428571428571429
